In [ ]:
#keypoint_proposal.py里的_preprocess把masks分割了，并且调整其他层大小为Dinov2所接受的分辨率大小，并且借助的仿真的三维点云图
import matplotlib.pyplot as plt
import torch
import numpy as np
import open3d as o3d
import yaml

# 导入你项目里的超级包装盒
from environment import ReKepOGEnv

# 读取 config.yaml 总图纸
with open('/home/clt/Rekep/configs/config.yaml', 'r') as f:
    full_config = yaml.safe_load(f)

# 【核心修复】：ReKepOGEnv 只认识 'env' 这个抽屉里的配置
env_config = full_config['env']

# 指定场景文件路径
scene_file = "/home/clt/Rekep/configs/og_scene_file_red_pen.json"

# 实例化 ReKepOGEnv，把抽屉配置递给它
env = ReKepOGEnv(config=env_config, scene_file=scene_file, verbose=True)

In [ ]:
# 重置环境（机器人会自动把手挪开，防止挡住相机视线）
env.reset()

In [ ]:
# 向所有相机发号施令，收取观测数据字典
obs_dict = env.get_cam_obs()

# 根据 config.yaml，提取 0号相机(vlm_camera) 的数据进行可视化
data = obs_dict[0]

#此时，上一步已经完成，摄像头的各个数据字典都有了
#接下来是对data中的seg即物品分割层进行处理，因为在real中，模型一次识别一个物体后输出一层真值图；
#而仿真的seg图层却是为不同物体赋予不同id的混合层，因此对于seg要进行处理得到多层二值图
#同时，仿真中的点云图points并非由深度图depth计算而来，而是仿真引擎提供的（控制变量这一块），点云图有图上各个点的三维坐标



In [1]:
##仅仅示例原理，处理分割图

import torch
import numpy as np

# 原始标签图（模拟你的 masks）
masks = torch.tensor([
    [0, 0, 1],
    [0, 2, 2]
], dtype=torch.int64)

print("原始 masks: ")
print(masks)

# 1. 找到所有出现过的标签 id
uids = np.unique(masks.numpy())
print("\nnp.unique 得到的 uids: ", uids)  # 预期: [0 1 2]

# 2. 按照每个 uid 生成二值掩码
masks_list = [masks == uid for uid in uids]

# 3. 逐个打印每个 uid 对应的二值掩码
for uid, m in zip(uids, masks_list):
    print(f"\nuid = {uid} 对应的二值掩码: ")
    print(m)

原始 masks: 
tensor([[0, 0, 1],
        [0, 2, 2]])

np.unique 得到的 uids:  [0 1 2]

uid = 0 对应的二值掩码: 
tensor([[ True,  True, False],
        [ True, False, False]])

uid = 1 对应的二值掩码: 
tensor([[False, False,  True],
        [False, False, False]])

uid = 2 对应的二值掩码: 
tensor([[False, False, False],
        [False,  True,  True]])
